# R1: Zero-shot p(True)

**Goal:** Measure how much a model trusts an answer by asking it
"Is the possible answer (A) True or (B) False?" and reading the
probability of its next token. Uncertainty = 1 − p(True).

**Day 1: Token diagnosis.** Before measuring p(True), find out *which token*
each model actually wants to say after the prompt ("True", " True", "A", "(", …),
and whether those candidates are single tokens.

Models: SmolLM2-135M-Instruct, Qwen2.5-0.5B-Instruct, Qwen2.5-1.5B-Instruct.

## Setup

In [2]:
import gc
import torch
from probly.representer.sampler.huggingface import load_model

torch.manual_seed(42)

MODELS = [
    "HuggingFaceTB/SmolLM2-135M-Instruct",
    "Qwen/Qwen2.5-0.5B-Instruct",
    "Qwen/Qwen2.5-1.5B-Instruct",
]

print("torch", torch.__version__)
print("models:", len(MODELS))

torch 2.10.0
models: 3


## Implementation

### 1. The p(True) prompt
Zero-shot prompt from LM-Polygraph (`stat_calculators/prompt.py`, `PromptCalculator`).

In [3]:
PROMPT = (
    "Question: {q}\n"
    " Possible answer: {a}\n"
    " Is the possible answer:\n"
    " (A) True\n"
    " (B) False\n"
    " The possible answer is:"
)

prompt = PROMPT.format(q="What is the capital of France?", a="Paris")
print(prompt)

Question: What is the capital of France?
 Possible answer: Paris
 Is the possible answer:
 (A) True
 (B) False
 The possible answer is:


### 2. Load a model and inspect its tokenizer
Start with the smallest model (SmolLM2-135M) to try things out quickly.

In [4]:
model, tok = load_model(MODELS[0])
model.eval()

n_params = sum(p.numel() for p in model.parameters())
print("model:", MODELS[0])
print("type:", type(model).__name__)
print(f"parameters: {n_params / 1e6:.0f} million")
print("dtype:", model.dtype)
print("vocabulary size:", len(tok))

Loading weights:   0%|          | 0/272 [00:00<?, ?it/s]

model: HuggingFaceTB/SmolLM2-135M-Instruct
type: LlamaForCausalLM
parameters: 135 million
dtype: torch.bfloat16
vocabulary size: 49152


In [5]:
example = "The capital of France is Paris. True"
ids = tok.encode(example, add_special_tokens=False)
pieces = [tok.decode([i]) for i in ids]

print("token ids:", ids)
print("pieces:   ", pieces)
print("number of tokens:", len(ids))

token ids: [504, 3575, 282, 4649, 314, 7042, 30, 3635]
pieces:    ['The', ' capital', ' of', ' France', ' is', ' Paris', '.', ' True']
number of tokens: 8


### 3. Apply the chat template
Instruct models expect a conversation format. We wrap the prompt as a user message
and open the assistant's turn, so the model's next token is its answer.

In [6]:
msgs = [{"role": "user", "content": prompt}]
text = tok.apply_chat_template(msgs, tokenize=False, add_generation_prompt=True)

print(text)
print("-" * 60)
print("characters:", len(text), "| tokens:", len(tok.encode(text, add_special_tokens=False)))

<|im_start|>system
You are a helpful AI assistant named SmolLM, trained by Hugging Face<|im_end|>
<|im_start|>user
Question: What is the capital of France?
 Possible answer: Paris
 Is the possible answer:
 (A) True
 (B) False
 The possible answer is:<|im_end|>
<|im_start|>assistant

------------------------------------------------------------
characters: 283 | tokens: 66


### 4. Next-token probabilities
One forward pass over the prompt; read the model's prediction for the token
right after "assistant", and list the 10 most likely candidates.

In [7]:
ids = tok(text, return_tensors="pt", add_special_tokens=False)
print("input shape: ", tuple(ids["input_ids"].shape))

with torch.no_grad():
    out = model(**ids)
print("logits shape:", tuple(out.logits.shape))

logits = out.logits[0, -1].float()
probs = torch.softmax(logits, dim=-1)
print("probabilities sum to:", round(probs.sum().item(), 4))

print("\n   id     token          probability")
top = torch.topk(probs, 10)
for p, i in zip(top.values, top.indices):
    print(f"{i.item():>7}  {tok.decode([i])!r:<14} {p.item():.4f}")

input shape:  (1, 66)
logits shape: (1, 66, 49152)
probabilities sum to: 1.0

   id     token          probability
     24  '('            0.3844
    504  'The'          0.2332
   2787  'True'         0.1414
   4245  'False'        0.0858
  10539  'Yes'          0.0316
     50  'B'            0.0149
   1589  'It'           0.0062
   1348  'This'         0.0055
     49  'A'            0.0055
  11192  'Based'        0.0048


### 5. Candidate answer tokens
Which of the possible "answer" strings are single tokens, and how likely is each
as the next token?

In [8]:
CANDIDATES = ["True", " True", "False", " False",
              "A", " A", "B", " B",
              "(A)", " (A)", "(", "Yes", "No"]

print(f"{'string':<9} {'ids':<16} {'pieces':<22} {'single':<7} P(first piece)")
for s in CANDIDATES:
    tid = tok.encode(s, add_special_tokens=False)
    pieces = [tok.decode([t]) for t in tid]
    print(f"{s!r:<9} {str(tid):<16} {str(pieces):<22} {str(len(tid) == 1):<7} {probs[tid[0]].item():.4f}")

string    ids              pieces                 single  P(first piece)
'True'    [2787]           ['True']               True    0.1414
' True'   [3635]           [' True']              True    0.0000
'False'   [4245]           ['False']              True    0.0858
' False'  [4178]           [' False']             True    0.0000
'A'       [49]             ['A']                  True    0.0055
' A'      [330]            [' A']                 True    0.0000
'B'       [50]             ['B']                  True    0.0149
' B'      [389]            [' B']                 True    0.0000
'(A)'     [24, 49, 25]     ['(', 'A', ')']        False   0.3844
' (A)'    [365, 49, 25]    [' (', 'A', ')']       False   0.0018
'('       [24]             ['(']                  True    0.3844
'Yes'     [10539]          ['Yes']                True    0.0316
'No'      [5230]           ['No']                 True    0.0043


### 6. All three models
Same diagnosis for SmolLM2-135M, Qwen2.5-0.5B and Qwen2.5-1.5B:
correct vs wrong answer (Paris / Berlin), with vs without chat template.

In [10]:
def next_token_probs(model, tok, question, answer, use_chat_template=True):
    """Probability of every vocabulary token as the next token after the p(True) prompt."""
    prompt = PROMPT.format(q=question, a=answer)
    if use_chat_template:
        text = tok.apply_chat_template([{"role": "user", "content": prompt}],
                                       tokenize=False, add_generation_prompt=True)
    else:
        text = prompt
    ids = tok(text, return_tensors="pt", add_special_tokens=False)
    with torch.no_grad():
        logits = model(**ids).logits[0, -1].float()
    return torch.softmax(logits, dim=-1)


def top_tokens(probs, tok, k=8):
    """The k most likely next tokens as (token, probability) pairs."""
    top = torch.topk(probs, k)
    return [(tok.decode([i]), round(p.item(), 3)) for p, i in zip(top.values, top.indices)]


def key_probs(probs, tok, strings=("True", " True", "False", " False", "A", " A", "B", " B", "(")):
    """Probability of each candidate string, if it is a single token (else None)."""
    out = {}
    for s in strings:
        tid = tok.encode(s, add_special_tokens=False)
        out[s] = round(probs[tid[0]].item(), 3) if len(tid) == 1 else None
    return out

In [11]:
del model, tok          # free SmolLM from the earlier steps
gc.collect()

Q = "What is the capital of France?"
results = {}

for name in MODELS:
    model, tok = load_model(name)
    model.eval()
    print("=" * 90)
    print(name)

    print("\n  single-token check:")
    for s in CANDIDATES:
        tid = tok.encode(s, add_special_tokens=False)
        print(f"    {s!r:<9} ids={str(tid):<18} single={len(tid) == 1}")

    for chat in [True, False]:
        for answer in ["Paris", "Berlin"]:
            probs = next_token_probs(model, tok, Q, answer, use_chat_template=chat)
            results[(name, chat, answer)] = {"top": top_tokens(probs, tok),
                                             "key": key_probs(probs, tok)}
            print(f"\n  chat_template={chat}  answer={answer}")
            print("    top-8:", results[(name, chat, answer)]["top"])
            print("    key:  ", results[(name, chat, answer)]["key"])

    del model, tok
    gc.collect()

Loading weights:   0%|          | 0/272 [00:00<?, ?it/s]

HuggingFaceTB/SmolLM2-135M-Instruct

  single-token check:
    'True'    ids=[2787]             single=True
    ' True'   ids=[3635]             single=True
    'False'   ids=[4245]             single=True
    ' False'  ids=[4178]             single=True
    'A'       ids=[49]               single=True
    ' A'      ids=[330]              single=True
    'B'       ids=[50]               single=True
    ' B'      ids=[389]              single=True
    '(A)'     ids=[24, 49, 25]       single=False
    ' (A)'    ids=[365, 49, 25]      single=False
    '('       ids=[24]               single=True
    'Yes'     ids=[10539]            single=True
    'No'      ids=[5230]             single=True

  chat_template=True  answer=Paris
    top-8: [('(', 0.384), ('The', 0.233), ('True', 0.141), ('False', 0.086), ('Yes', 0.032), ('B', 0.015), ('It', 0.006), ('This', 0.005)]
    key:   {'True': 0.141, ' True': 0.0, 'False': 0.086, ' False': 0.0, 'A': 0.005, ' A': 0.0, 'B': 0.015, ' B': 0.0, '(': 0.38

Loading weights:   0%|          | 0/290 [00:00<?, ?it/s]

Qwen/Qwen2.5-0.5B-Instruct

  single-token check:
    'True'    ids=[2514]             single=True
    ' True'   ids=[3007]             single=True
    'False'   ids=[4049]             single=True
    ' False'  ids=[3557]             single=True
    'A'       ids=[32]               single=True
    ' A'      ids=[362]              single=True
    'B'       ids=[33]               single=True
    ' B'      ids=[425]              single=True
    '(A)'     ids=[4346, 8]          single=False
    ' (A)'    ids=[320, 32, 8]       single=False
    '('       ids=[7]                single=True
    'Yes'     ids=[9454]             single=True
    'No'      ids=[2753]             single=True

  chat_template=True  answer=Paris
    top-8: [('(A', 0.653), ('The', 0.187), ('(B', 0.054), ('A', 0.032), ('(', 0.017), ('(C', 0.008), ('Paris', 0.007), ('Correct', 0.004)]
    key:   {'True': 0.002, ' True': 0.0, 'False': 0.0, ' False': 0.0, 'A': 0.032, ' A': 0.0, 'B': 0.004, ' B': 0.0, '(': 0.017}

  chat_

Loading weights:   0%|          | 0/338 [00:00<?, ?it/s]

Qwen/Qwen2.5-1.5B-Instruct

  single-token check:
    'True'    ids=[2514]             single=True
    ' True'   ids=[3007]             single=True
    'False'   ids=[4049]             single=True
    ' False'  ids=[3557]             single=True
    'A'       ids=[32]               single=True
    ' A'      ids=[362]              single=True
    'B'       ids=[33]               single=True
    ' B'      ids=[425]              single=True
    '(A)'     ids=[4346, 8]          single=False
    ' (A)'    ids=[320, 32, 8]       single=False
    '('       ids=[7]                single=True
    'Yes'     ids=[9454]             single=True
    'No'      ids=[2753]             single=True

  chat_template=True  answer=Paris
    top-8: [('(A', 0.866), ('The', 0.091), ('(B', 0.02), ('A', 0.018), ('True', 0.002), ('(True', 0.001), ('(C', 0.0), (' (', 0.0)]
    key:   {'True': 0.002, ' True': 0.0, 'False': 0.0, ' False': 0.0, 'A': 0.018, ' A': 0.0, 'B': 0.0, ' B': 0.0, '(': 0.0}

  chat_template=Tr

#### Observations (all three models, "What is the capital of France?")

**Tokenization**
- All word candidates (`True`, `False`, `A`, `B`, `Yes`, `No`, with and without leading space) are single tokens in both tokenizers.
- `(A)` is split differently: SmolLM → `(` + `A` + `)`; Qwen → `(A` + `)`.
  So for **Qwen, `(A` and `(B` are single tokens** and carry the verdict directly.
  For **SmolLM, the verdict hides behind `(`**, which is shared by (A) and (B) and therefore uninformative.
- At the start of the assistant turn (chat template), models use tokens **without** a leading space
  (`True`, `(A`); in the plain prompt (no template) they continue mid-line and use tokens **with** a space (`' ('`, `' True'`).

**With chat template**
- **SmolLM-135M:** top tokens `(` (0.38), `The` (0.23), `True` (0.14), `False` (0.09).
  True vs False barely changes between Paris and Berlin (≈0.62 vs ≈0.56) → almost no signal.
  The letter route even points the wrong way (B > A for Paris).
- **Qwen-0.5B:** `(A` dominates (0.65 for Paris, 0.45 for Berlin); `(B` rises from 0.05 to 0.17 for Berlin.
  → Signal exists, but the model still leans "True" for the wrong answer.
- **Qwen-1.5B:** Paris → `(A` 0.87; Berlin → `(B` 0.97. → Clear, almost perfect separation.

**Without chat template**
- Both Qwen models put 97–99 % on `' ('` → verdict hidden behind an ambiguous token → **chat template needed for Qwen**.
- SmolLM without template: p(True) 0.41 (Paris) vs 0.30 (Berlin), a larger gap than with the template, but it leans "False" even for the correct answer.

**Takeaway for Day 2:** use per-model token groups and keep only strings that are single tokens
for that tokenizer (Qwen: `(A`/`A` vs `(B`/`B`; SmolLM: `True` vs `False`).

### 7. Preview: normalized p(True) with per-model token groups
Groups contain only strings that are single tokens for the model's tokenizer.
p(True) = P(True group) / (P(True group) + P(False group)).

In [12]:
TRUE_STRINGS  = ["True", " True", "A", " A", "(A", " (A"]
FALSE_STRINGS = ["False", " False", "B", " B", "(B", " (B"]

def single_token_ids(tok, strings):
    """Token ids of the strings that are exactly one token for this tokenizer."""
    ids = {}
    for s in strings:
        tid = tok.encode(s, add_special_tokens=False)
        if len(tid) == 1:
            ids[s] = tid[0]
    return ids

summary = []
for name in MODELS:
    model, tok = load_model(name)
    model.eval()
    t_ids = single_token_ids(tok, TRUE_STRINGS)
    f_ids = single_token_ids(tok, FALSE_STRINGS)
    print(name)
    print("  True group: ", list(t_ids))
    print("  False group:", list(f_ids))

    for chat in [True, False]:
        for answer in ["Paris", "Berlin"]:
            probs = next_token_probs(model, tok, Q, answer, use_chat_template=chat)
            p_t = sum(probs[i].item() for i in t_ids.values())
            p_f = sum(probs[i].item() for i in f_ids.values())
            summary.append({"model": name.split("/")[1], "chat": chat, "answer": answer,
                            "P(True grp)": round(p_t, 3), "P(False grp)": round(p_f, 3),
                            "covered": round(p_t + p_f, 3),
                            "p_true_norm": round(p_t / (p_t + p_f), 3)})
    del model, tok
    gc.collect()

import pandas as pd
pd.DataFrame(summary)

Loading weights:   0%|          | 0/272 [00:00<?, ?it/s]

HuggingFaceTB/SmolLM2-135M-Instruct
  True group:  ['True', ' True', 'A', ' A']
  False group: ['False', ' False', 'B', ' B']


Loading weights:   0%|          | 0/290 [00:00<?, ?it/s]

Qwen/Qwen2.5-0.5B-Instruct
  True group:  ['True', ' True', 'A', ' A', '(A']
  False group: ['False', ' False', 'B', ' B', '(B']


Loading weights:   0%|          | 0/338 [00:00<?, ?it/s]

Qwen/Qwen2.5-1.5B-Instruct
  True group:  ['True', ' True', 'A', ' A', '(A']
  False group: ['False', ' False', 'B', ' B', '(B']


,model,chat,answer,P(True grp),P(False grp),covered,p_true_norm
0,SmolLM2-135M-Instruct,True,Paris,0.147,0.101,0.248,0.593
1,SmolLM2-135M-Instruct,True,Berlin,0.165,0.142,0.307,0.538
2,SmolLM2-135M-Instruct,False,Paris,0.104,0.150,0.255,0.410
3,SmolLM2-135M-Instruct,False,Berlin,0.077,0.181,0.259,0.299
4,Qwen2.5-0.5B-Instruct,True,Paris,0.688,0.058,0.745,0.923
5,Qwen2.5-0.5B-Instruct,True,Berlin,0.486,0.184,0.670,0.726
6,Qwen2.5-0.5B-Instruct,False,Paris,0.076,0.001,0.077,0.985
7,Qwen2.5-0.5B-Instruct,False,Berlin,0.001,0.015,0.016,0.051
8,Qwen2.5-1.5B-Instruct,True,Paris,0.886,0.020,0.906,0.977
9,Qwen2.5-1.5B-Instruct,True,Berlin,0.001,0.991,0.991,0.001


## Results: Day 1 summary

### Token groups per model
Rule: candidate strings are kept only if they are a **single token** for the model's tokenizer.

| Model | True group | False group | Verdict token in practice |
|---|---|---|---|
| SmolLM2-135M | `True`, ` True`, `A`, ` A` | `False`, ` False`, `B`, ` B` | mostly `(` (ambiguous) or `True`/`False` |
| Qwen2.5-0.5B | `True`, ` True`, `A`, ` A`, `(A` | `False`, ` False`, `B`, ` B`, `(B` | `(A` / `(B` |
| Qwen2.5-1.5B | `True`, ` True`, `A`, ` A`, `(A` | `False`, ` False`, `B`, ` B`, `(B` | `(A` / `(B` |

### Normalized p(True), one question (France: Paris ✓ / Berlin ✗)

| Model | Template | p(True) Paris | p(True) Berlin | Gap | Coverage |
|---|---|---|---|---|---|
| SmolLM2-135M | yes | 0.593 | 0.538 | 0.06 | ~0.25–0.31 |
| SmolLM2-135M | no | 0.410 | 0.299 | 0.11 | ~0.26 |
| Qwen2.5-0.5B | yes | 0.923 | 0.726 | 0.20 | ~0.67–0.75 |
| Qwen2.5-0.5B | no | 0.985 | 0.051 | 0.93 | **0.02–0.08** |
| Qwen2.5-1.5B | yes | 0.977 | 0.001 | 0.98 | ~0.91–0.99 |
| Qwen2.5-1.5B | no | 0.992 | 0.001 | 0.99 | **0.01–0.02** |

*Coverage = P(True group) + P(False group): how much of the model's next-token probability the two groups capture.*

### What works / What does not
- ✅ Qwen tokenizes `(A` / `(B` as single tokens → the verdict can be read directly.
- ✅ With the chat template, Qwen-1.5B separates right and wrong almost perfectly; Qwen-0.5B shows a weaker signal.
- ❌ SmolLM-135M shows almost no signal; its preferred first token `(` is shared by (A) and (B).
- ⚠️ Without the template, normalized values look strong but rest on 1–8 % of the probability mass → unreliable.
- ⚠️ Raw p(True) (single "True" token, LM-Polygraph style) would be near 0 for Qwen, because Qwen answers with `(A`, not `True`.

### Plan for Day 2
- Use the chat template; per-model single-token groups as above.
- Run the sanity test (10 questions × correct/wrong) for normalized and raw, all 3 models.


### Questions for tutors
- When normalization rests on very little probability mass (low coverage), should we trust it, flag it, or fall back to another method?

# Day 2: Normalized p(True) and sanity test

### 1. Normalization: p_true_from_logits
Compare only the "True" tokens against the "False" tokens and ignore every
other token in the vocabulary.

In [18]:
def p_true_from_logits(logits, true_ids, false_ids):
    """Normalized p(True): P(True group) / (P(True group) + P(False group))."""
    lt = torch.logsumexp(logits[true_ids], 0)
    lf = torch.logsumexp(logits[false_ids], 0)
    return torch.softmax(torch.stack([lt, lf]), 0)[0].item()



# Test on a made-up 5-token vocabulary
fake = torch.tensor([2.0, 1.0, 0.0, -1.0, 5.0])
true_ids, false_ids = [0, 1], [2, 3]

print("normalized (logits way):", round(p_true_from_logits(fake, true_ids, false_ids), 4))

probs = torch.softmax(fake, 0)
p_t, p_f = probs[true_ids].sum(), probs[false_ids].sum()
print("normalized (probs way): ", round((p_t / (p_t + p_f)).item(), 4))


normalized (logits way): 0.8808
normalized (probs way):  0.8808


### 2. The p_true function (draft for common/reflexive.py, with R2)
`p_true(model, tok, question, answer, brainstorm=None)` → normalized p(True), as in the plan.
`true_logit(...)` → the raw "True" logit, as LM-Polygraph does, for comparison.
The `brainstorm` argument is part of the plan's signature; its prompt is added by R2.

In [15]:
def next_token_logits(model, tok, question, answer):
    """Fill the zero-shot prompt, apply the chat template, return the logits for the next token."""
    prompt = PROMPT.format(q=question, a=answer)
    text = tok.apply_chat_template([{"role": "user", "content": prompt}],
                                   tokenize=False, add_generation_prompt=True)
    ids = tok(text, return_tensors="pt", add_special_tokens=False)
    with torch.no_grad():
        return model(**ids).logits[0, -1].float()


def token_groups(tok):
    """True / False token ids for this tokenizer: only strings that are single tokens (Day 1 finding)."""
    true_ids = list(single_token_ids(tok, TRUE_STRINGS).values())
    false_ids = list(single_token_ids(tok, FALSE_STRINGS).values())
    return true_ids, false_ids


def p_true(model, tok, question, answer, brainstorm=None):
    """Normalized zero-shot p(True) for one question-answer pair."""
    if brainstorm is not None:
        raise NotImplementedError("Brainstorm prompt: added by R2")
    logits = next_token_logits(model, tok, question, answer)
    true_ids, false_ids = token_groups(tok)
    return p_true_from_logits(logits, true_ids, false_ids)


def true_logit(model, tok, question, answer):
    """Raw logit of the single token 'True', as LM-Polygraph does."""
    logits = next_token_logits(model, tok, question, answer)
    true_id = tok.encode("True", add_special_tokens=False)[0]
    return logits[true_id].item()

# Day 3: Real data (Qwen2.5-0.5B, 50 TriviaQA questions)

### 1. Load B1's 50-question file

In [16]:
import json, time
from pathlib import Path

DATA = Path("../data/qwen0.5b/triviaqa_50.jsonl")
rows = [json.loads(line) for line in DATA.read_text().splitlines() if line.strip()]

print("questions:", len(rows))
print("model in file:", rows[0]["model"])
print("accuracy of greedy answers:", sum(r["correct"] for r in rows) / len(rows))

questions: 50
model in file: Qwen/Qwen2.5-0.5B-Instruct
accuracy of greedy answers: 0.24


### 2. Normalized zero-shot p(True) on the main (greedy) answers

In [17]:
MAIN_MODEL = "Qwen/Qwen2.5-0.5B-Instruct"
model, tok = load_model(MAIN_MODEL)
model.eval()

p_true_scores, failed, times = [], [], []
for r in rows:
    start = time.perf_counter()
    try:
        p = p_true(model, tok, r["question"], r["greedy_answer"])
        failed.append(False)
    except Exception as e:
        print("failed on id", r["id"], ":", e)
        p = 0.0                      # → uncertainty 1.0, as the contract requires
        failed.append(True)
    times.append(time.perf_counter() - start)
    p_true_scores.append(p)

print("done:", len(p_true_scores), "| failed:", sum(failed))
print("seconds per question:", round(sum(times) / len(times), 3))

right = [p for p, r in zip(p_true_scores, rows) if r["correct"]]
wrong = [p for p, r in zip(p_true_scores, rows) if not r["correct"]]
print("mean p(True) | correct answers:", round(sum(right) / len(right), 3))
print("mean p(True) | wrong answers:  ", round(sum(wrong) / len(wrong), 3))

Loading weights:   0%|          | 0/290 [00:00<?, ?it/s]

done: 50 | failed: 0
seconds per question: 0.721
mean p(True) | correct answers: 0.9
mean p(True) | wrong answers:   0.855


# Day 2 (continued): Sanity test with R2's run_sanity()

In [19]:
import sys
from pathlib import Path

LLM_DIR = Path.cwd().parent          # experiments/llm-probly
sys.path.insert(0, str(LLM_DIR))
from common.sanity import run_sanity, save_sanity, SANITY

model, tok = load_model("Qwen/Qwen2.5-0.5B-Instruct")
out = p_true(model, tok, "What is the capital of France?", "Paris")
print(type(out))
print(out)

Loading weights:   0%|          | 0/290 [00:00<?, ?it/s]

<class 'float'>
0.9227984547615051


### Sanity test (R2's `run_sanity()`, common/sanity.py)

10 easy questions, each scored twice: once with the correct answer, once with a wrong one
(e.g. Paris vs Berlin). A working method gives higher confidence to the correct answers.

- **Methods:** normalized zero-shot p(True) (`p_true`, per-model token groups) and
  raw "True" logit (`true_logit`, as in LM-Polygraph)
- **Models:** SmolLM2-135M, Qwen2.5-0.5B, Qwen2.5-1.5B (all Instruct, with chat template)
- **Metrics:** AUROC (correct = positive; 1.0 = perfect, 0.5 = chance),
  gap (mean confidence correct − wrong), n_distinct (number of different confidence values)
- **Note:** raw values are logits, so only AUROC is comparable between normalized and raw.
- **Output:** `results/sanity/p_true_zeroshot_{norm,raw}_<model>.json`

In [21]:
import gc, time
import torch

SANITY_DIR = LLM_DIR / "results" / "sanity"
SANITY_MODELS = [
    "HuggingFaceTB/SmolLM2-135M-Instruct",
    "Qwen/Qwen2.5-0.5B-Instruct",
    "Qwen/Qwen2.5-1.5B-Instruct",
]

sanity_results = {}
for name in SANITY_MODELS:
    model, tok = load_model(name)

    # Wrappers: run_sanity calls confidence_fn(question, answer, samples)
    def conf_norm(q, a, s=None):
        return p_true(model, tok, q, a)

    def conf_raw(q, a, s=None):
        return float(true_logit(model, tok, q, a))

    for method, fn in [("p_true_zeroshot_norm", conf_norm),
                       ("p_true_zeroshot_raw", conf_raw)]:
        t0 = time.time()
        res = run_sanity(fn)
        res["seconds_total"] = round(time.time() - t0, 2)
        path = save_sanity(res, method, name, folder=SANITY_DIR)
        sanity_results[(name.split("/")[-1], method)] = res
        print(f"{name.split('/')[-1]:28s} {method:22s} "
              f"AUROC={res['auroc']:.3f}  gap={res['gap']:+.3f}  "
              f"distinct={res['n_distinct']:2d}  -> {path.name}")

    del model, tok
    gc.collect()

Loading weights:   0%|          | 0/272 [00:00<?, ?it/s]

SmolLM2-135M-Instruct        p_true_zeroshot_norm   AUROC=0.400  gap=-0.021  distinct=20  -> p_true_zeroshot_norm_SmolLM2-135M-Instruct.json
SmolLM2-135M-Instruct        p_true_zeroshot_raw    AUROC=0.465  gap=-0.075  distinct= 6  -> p_true_zeroshot_raw_SmolLM2-135M-Instruct.json


Loading weights:   0%|          | 0/290 [00:00<?, ?it/s]

Qwen2.5-0.5B-Instruct        p_true_zeroshot_norm   AUROC=0.700  gap=+0.061  distinct=20  -> p_true_zeroshot_norm_Qwen2.5-0.5B-Instruct.json
Qwen2.5-0.5B-Instruct        p_true_zeroshot_raw    AUROC=0.565  gap=+0.219  distinct=14  -> p_true_zeroshot_raw_Qwen2.5-0.5B-Instruct.json


Loading weights:   0%|          | 0/338 [00:00<?, ?it/s]

Qwen2.5-1.5B-Instruct        p_true_zeroshot_norm   AUROC=0.970  gap=+0.685  distinct=17  -> p_true_zeroshot_norm_Qwen2.5-1.5B-Instruct.json
Qwen2.5-1.5B-Instruct        p_true_zeroshot_raw    AUROC=0.910  gap=+2.050  distinct=17  -> p_true_zeroshot_raw_Qwen2.5-1.5B-Instruct.json


### Sanity test results

| Model | AUROC normalized | AUROC raw | gap normalized |
|---|---|---|---|
| SmolLM2-135M | 0.400 | 0.465 | −0.021 |
| Qwen2.5-0.5B | 0.700 | 0.565 | +0.061 |
| Qwen2.5-1.5B | **0.970** | 0.910 | +0.685 |

Runtime: ~4.5 min for all 3 models × 2 methods × 20 answers (MacBook Air, CPU/MPS).

**Findings**
- Zero-shot p(True) **works from 1.5B on** (AUROC 0.97), is **weak at 0.5B** (0.70, small gap),
  and gives **no signal at 135M** (around chance).
- **Normalized > raw** for both Qwen models (0.70 vs 0.57, 0.97 vs 0.91): measuring only the
  "True" token (LM-Polygraph) loses probability that Qwen puts on `(A`/`(B`.
- The group's main model (Qwen 0.5B) is in the weak range, so zero-shot p(True) results on the
  50 TriviaQA questions should be read with care.
- Caveat: only 10 questions (20 answers), so small AUROC differences are not meaningful.

---
# Day 3: p(True) on the 50 TriviaQA questions (score contract)

In [22]:
import json

DATA = LLM_DIR / "data" / "qwen0.5b" / "triviaqa_50.jsonl"
rows = [json.loads(line) for line in open(DATA)]
print(len(rows), "questions")
print("Keys:", list(rows[0].keys()))
print(rows[0])

print("\nExisting results in results/qwen0.5b/:")
for p in sorted((LLM_DIR / "results" / "qwen0.5b").glob("*")):
    print(" ", p.name)

50 questions
Keys: ['id', 'question', 'aliases', 'greedy_answer', 'greedy_ll_sum', 'greedy_ll_mean', 'samples', 'samples_ll_mean', 'model', 'correct_alias', 'correct', 'samples_correct']
{'id': 0, 'question': 'In April 1963, Lester Bowles Pearson became Prime Minister of which country?', 'aliases': ['Canada', 'Canadá', 'The Dominion of Canada', 'Commonwealth of Canada', 'Dominion of canada', 'ISO 3166-1:CA', 'Etymology of Canada', 'CANADA', 'Canadiophile', "Canada's", 'ᑲᓇᑕ', 'Canada/References', "America's top hat", 'Canada.', 'Cnada', 'Čanada', 'Canadian Federation', 'Kenadian', 'Canadialand', 'Xanada', 'Dominion of Canada', 'Canadaa'], 'greedy_answer': 'United Kingdom', 'greedy_ll_sum': -1.3600367307662964, 'greedy_ll_mean': -0.6800183653831482, 'samples': ['United Kingdom', 'United Kingdom', 'United Kingdom', 'United Kingdom', 'United Kingdom', 'Australia', 'United Kingdom', 'United Kingdom', 'UK', 'April 1963, Lester Bowles Pearson became Prime Minister of Canada.'], 'samples_ll_me

In [23]:
import time, math

MODEL_NAME = "Qwen/Qwen2.5-0.5B-Instruct"
assert all(r["model"] == MODEL_NAME for r in rows), "data file is from a different model"

model, tok = load_model(MODEL_NAME)

ids, unc, failed, times = [], [], [], []
for r in rows:
    t0 = time.time()
    try:
        p = float(p_true(model, tok, r["question"], r["greedy_answer"]))
        if math.isnan(p) or not (0.0 <= p <= 1.0):
            raise ValueError(f"bad p_true value {p}")
        u, f = 1.0 - p, False              # contract rule 1: higher = more uncertain
    except Exception as e:
        print(f"id {r['id']} failed: {e}")
        u, f = 1.0, True                   # contract rule 3
    times.append(time.time() - t0)
    ids.append(r["id"]); unc.append(u); failed.append(f)

result = {
    "method": "p_true_zeroshot_norm",
    "model": MODEL_NAME,
    "answer": "greedy",
    "ids": ids,
    "uncertainty": unc,
    "failed": failed,
    "seconds_per_question": round(sum(times) / len(times), 3),
}
OUT = LLM_DIR / "results" / "qwen0.5b" / "p_true_zeroshot_norm.json"
with open(OUT, "w") as fh:
    json.dump(result, fh, indent=2)

# Check the saved file against the contract
saved = json.load(open(OUT))
assert saved["ids"] == [r["id"] for r in rows]                      # rule 5: same ids, same order
assert len(saved["uncertainty"]) == len(saved["failed"]) == len(rows)  # rule 3: nothing missing
assert all(0.0 <= u <= 1.0 for u in saved["uncertainty"])
print(f"saved {OUT.name} | {len(ids)} questions | failed: {sum(failed)} | "
      f"{saved['seconds_per_question']} s/question")

Loading weights:   0%|          | 0/290 [00:00<?, ?it/s]

saved p_true_zeroshot_norm.json | 50 questions | failed: 0 | 0.786 s/question
